# PART 2 défi 1 : Pytorch
**Groupe 2 :**
- **SAOUDI Lydia**
- **NKRUMAH-BUADII Cornelius**
- **LEYENBERGER Emma**
- **ENGIN Belfin**
- **HUANG Flora**
- **LAMLOUH Awataf**

### Objectif :

L'objectif est le même, développer et valider un modèle d'apprentissage automatique capable de prédire la survenue d'une bactériémie chez un patient (1 = oui / 0 = non). Dans la première partie du défi, nous avons prédit la bactériémie avec des modèles classiques de scikit-learn. Ici, nous utilisons PyTorch pour construire un réseau de neurones et comparer ses résultats aux autres modèles.

Pourquoi PyTorch ? :
- **Un modèle sur mesure** : nous choisissons nous-mêmes les couches du réseau (nombre, taille, `Dropout`), ce que les modèles classiques ne permettent pas.
- **Un apprentissage contrôlé** : nous écrivons la boucle d'entraînement. Nous pouvons ainsi mesurer le F1 à chaque époque et garder la meilleure.
- **Un traitement des cas rares** : les patients avec bactériémie sont peu nombreux (environ 8 %). Avec `pos_weight`, nous donnons plus de poids à leurs erreurs.
- **Une comparaison utile** : tester un réseau de neurones sur les mêmes données et le même préprocesseur permet de voir s'il apporte quelque chose de plus.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# Charger les données
data = pd.read_csv('bacteriemie.csv.gz')
data['SEX'] = data['SEX'].astype(object)
pd.set_option('display.max_columns', None)
data.sample(4)

,ID,SEX,AGE,MCV,HGB,HCT,PLT,MCH,MCHC,RDW,MPV,LYM,MONO,EOS,BASO,NT,APTT,FIB,SODIUM,POTASS,CA,PHOS,MG,CREA,BUN,HS,GBIL,TP,ALB,AMY,PAMY,LIP,CHE,AP,ASAT,ALAT,GGT,LDH,CK,GLU,TRIG,CHOL,CRP,BASOR,EOSR,LYMR,MONOR,NEU,NEUR,PDW,RBC,WBC,BloodCulture
1417,6101,1,83,92.9,13.1,40.9,167.0,29.5,31.8,13.2,10.5,1.2,0.4,0.0,0.0,119.0,NaN,NaN,142.0,4.37,NaN,NaN,NaN,1.55,20.2,NaN,0.35,NaN,NaN,167.0,NaN,63.0,NaN,110.0,28.0,15.0,15.0,221.0,272.0,95.0,NaN,NaN,0.40,0.000000,0.000000,13.953488,4.651163,7.0,81.395349,12.0,4.5,8.73,no
5142,25238,1,68,81.0,13.1,40.7,157.0,25.9,32.2,15.4,11.4,NaN,NaN,NaN,0.1,94.0,32.6,1048.0,140.0,5.00,2.38,2.91,1.38,3.57,107.3,19.8,2.04,76.7,29.3,43.0,32.0,121.0,3.33,121.0,180.0,79.0,79.0,348.0,3327.0,214.0,153.0,134.0,36.89,NaN,NaN,NaN,NaN,NaN,NaN,14.4,4.8,24.08,yes
2579,11931,2,81,93.8,10.7,31.7,140.0,31.2,33.1,14.8,10.9,0.8,1.0,0.3,0.0,106.0,31.8,545.0,139.0,4.30,2.21,1.02,0.89,0.98,18.6,6.7,0.68,66.1,34.7,34.0,6.0,9.0,3.12,59.0,23.0,13.0,32.0,166.0,51.0,NaN,120.0,149.0,4.46,0.000000,2.702703,7.207207,9.009009,9.0,81.081081,11.6,3.4,11.25,no
317,1182,1,67,87.9,9.6,31.6,559.0,26.4,30.1,15.4,10.8,0.8,0.4,0.0,0.1,64.0,46.0,835.0,132.0,4.92,2.07,1.48,1.05,2.94,53.2,NaN,0.70,48.1,19.2,74.0,NaN,18.0,3.30,173.0,20.0,13.0,219.0,369.0,41.0,125.0,NaN,NaN,26.10,0.917431,0.000000,7.339450,3.669725,9.6,88.073394,12.8,3.5,10.98,no


L'analyse est la même que dans la partie précédente. Dans cette partie, nous appliquons le même processus de prétraitement avant de passer à la conversion des données en tenseurs.

In [3]:
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.pipeline import Pipeline

# Séparer les features (X) et la cible (y)
y = data['BloodCulture']
X = data.drop(['BloodCulture', 'ID'], axis=1)

# Diviser les données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, shuffle=True, random_state=42)

variables_catégorielles = ['SEX']
variables_quantitatives = X_train.columns.difference(variables_catégorielles).tolist()

On note qu'un réseau de neurone supporte mal les cases vides et les valeurs à des échelles très différentes. C'est pour cela qu'on applique les préprocesseurs suivants :
- IterativeImputer : remplit les cases vides en les devinant à partir des autres colonnes
- SimpleImputer : remplit avec la valeur la plus fréquente
- **RobustScaler** : met à la même échelle, sans se laisser gêner par les valeurs extrêmes
- OneHotEncoder : crée des colonnes 0/1 (une par valeur)
- **add_indicator=True** : ajoute une colonne 0/1 qui indique si la valeur était manquante (un examen non prescrit est une information utile)

In [4]:
# Pipeline pour les variables numériques
numeric_transformer = Pipeline(steps=[
    ('imputer', IterativeImputer(random_state=0, add_indicator=True)),
    ('scaler', RobustScaler())
])

# Pipeline pour les variables catégorielles
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="most_frequent")), # Impute avec la valeur la plus fréquente
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Assemblage avec ColumnTransformer
preprocesseur = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, variables_quantitatives),
        ('cat', categorical_transformer, variables_catégorielles)
    ]
)

In [5]:
# Séparer entraînement / validation AVANT le prétraitement
# (le préprocesseur ne doit pas voir les données de validation)
idx_tr, idx_val = train_test_split(
    np.arange(len(X_train)), test_size=0.2, stratify=y_train, random_state=42)

# Le préprocesseur apprend uniquement sur la partie entraînement
X_tr = preprocesseur.fit_transform(X_train.iloc[idx_tr])
X_val = preprocesseur.transform(X_train.iloc[idx_val])
X_test_preprocessed = preprocesseur.transform(X_test)

if hasattr(X_tr, "toarray"):
    X_tr = X_tr.toarray()
    X_val = X_val.toarray()
    X_test_preprocessed = X_test_preprocessed.toarray()

/usr/local/lib/python3.13/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


note : si le preprocesseur sort une matrice creuse (sparse matrix) du OneHotEncoder, `.toarray()`permet de garantir la conversion pour PyTorch.

In [6]:
# Encodage de la cible y
y_train = y_train.map({'yes': 1, 'no': 0})
y_test = y_test.map({'yes': 1, 'no': 0})
assert y_train.notna().all() and y_test.notna().all(), "BloodCulture doit contenir uniquement 'yes' ou 'no'"

In [7]:
# Conversion du test en tenseur PyTorch (utilisé pour l'évaluation finale)
X_test_tensor = torch.FloatTensor(X_test_preprocessed)

# Réseau de neurones PyTorch : viser le meilleur F1-score
Même objectif que pour les autres modèles : le meilleur **F1-score**, puis la meilleure **précision**.

In [8]:
from sklearn.metrics import f1_score, precision_score
import copy

torch.manual_seed(42)
np.random.seed(42)

# Séparation du set de données
# X_tr et X_val viennent du découpage fait avant le prétraitement (cellule 7)
y_tr, y_val = y_train.values[idx_tr], y_train.values[idx_val]

# Création du DataLoader
X_val_tensor = torch.FloatTensor(X_val)
train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr).reshape(-1, 1)),
    batch_size=64, shuffle=True, drop_last=True)
# batch de 64 + mélange ; drop_last évite un dernier batch d'un seul patient (BatchNorm plante)

# Optimisation du seuil

seuils = np.linspace(0.01, 0.99, 99)

def meilleur_seuil(y, proba):
    f1_par_seuil = [f1_score(y, (proba >= t).astype(int), zero_division=0) for t in seuils]
    seuil = seuils[np.argmax(f1_par_seuil)]
    precision = precision_score(y, (proba >= seuil).astype(int), zero_division=0)
    return seuil, max(f1_par_seuil), precision

## Le modèle

Choix d'un réseau petit : 3 couches cachées. <br>

Nous avons choisi un réseau volontairement petit car avec environ 600 patients positifs, un réseau plus grand risque d’apprendre les données par cœur.

Modèle : Entrée (variables après prétraitement, indicateurs de valeurs manquantes compris) → 64 → 32 → 16 → sortie (1 score)

| Élément | Rôle |
|---|---|
| `Linear` | Mélange les variables pour en créer de nouvelles. C'est là que le réseau apprend. |
| `BatchNorm1d` | Garde les valeurs à une échelle stable, pour un entraînement plus régulier. |
| `ReLU` | Remplace les valeurs négatives par 0. Sans elle, empiler des couches ne servirait à rien. |
| `Dropout(0.2)` | Éteint 20 % des neurones au hasard pendant l'entraînement, pour éviter d'apprendre les données par cœur. |

La dernière couche donne un seul score par patient. On ne l'entoure d'aucun autre élément. Ce score est ensuite transformé en probabilité avec `torch.sigmoid`.

On met le `Dropout` seulement après les deux premières couches cachées, pas juste avant la sortie.

In [9]:
class Modele(nn.Module):
    def __init__(self, n_entrees):  # __init__ construit les couches
        super().__init__()
        self.reseau = nn.Sequential(
            # Bloc 1 : colonnes d'entrée -> 64
            nn.Linear(n_entrees, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2), # évite l'overfitting

            # Bloc 2 : 64 -> 32
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Bloc 3 : 32 -> 16
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),

            # Sortie : un seul score par patient
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.reseau(x)

model = Modele(X_tr.shape[1])

# pos_weight : permet de donner plus d'importance aux cas positifs de bactériémie rare
ratio_brut = (y_tr == 0).sum() / (y_tr == 1).sum()
poids_ajuste = torch.tensor(np.sqrt(ratio_brut), dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=poids_ajuste) # mesure à quel point le modèle se trompe

# AdamW : l’optimiseur, qui modifie les réglages du réseau pour réduire l’erreur.
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

## Entraînement
À chaque époque, on calcule le F1 sur la validation (avec son meilleur seuil). On garde le modèle de l'époque qui a le meilleur F1, puis la meilleure précision en cas d'égalité.

Le `scheduler` divise la taille des pas par 2 quand le F1 ne progresse plus pendant 3 époques.

In [10]:
meilleur_score = (0, 0)
meilleur_etat, seuil_final = None, 0.5

for epoch in range(20):
    model.train()
    for batch_X, batch_y in train_loader:
        loss = criterion(model(batch_X), batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        proba_val = torch.sigmoid(model(X_val_tensor)).numpy().ravel()

    seuil, f1, precision = meilleur_seuil(y_val, proba_val)
    scheduler.step(f1)

    if (round(f1, 3), precision) > meilleur_score:
        meilleur_score = (round(f1, 3), precision)
        meilleur_etat = copy.deepcopy(model.state_dict())
        seuil_final = seuil

    if (epoch + 1) % 5 == 0:
        print(
            f'Époque {epoch+1} : F1 val = {f1:.3f} | Précision ='
            f' {precision:.3f}')
print(
    f'\nMeilleur modèle sélectionné : F1 val = {meilleur_score[0]:.3f} | Seuil ='
    f' {seuil_final:.2f}')

Époque 5 : F1 val = 0.380 | Précision = 0.303
Époque 10 : F1 val = 0.364 | Précision = 0.292
Époque 15 : F1 val = 0.364 | Précision = 0.318
Époque 20 : F1 val = 0.359 | Précision = 0.290

Meilleur modèle sélectionné : F1 val = 0.380 | Seuil = 0.37


## Évaluation finale sur le test
On recharge le meilleur modèle et on l'évalue une seule fois sur le test, avec le seuil choisi sur la validation.

In [11]:
model.load_state_dict(meilleur_etat)
model.eval()

with torch.no_grad():
  proba_test = torch.sigmoid(model(X_test_tensor)).numpy().ravel()

prediction = (proba_test >= seuil_final).astype(int)

print(
    f'\nF1 test = {f1_score(y_test, prediction):.3f} | Précision test ='
    f' {precision_score(y_test, prediction):.3f}'
)


F1 test = 0.315 | Précision test = 0.244


## Prédire de nouvelles données
Même chemin que pour le test : préparation des données, probabilité du réseau, puis seuil.

In [12]:
nouvelles_donnees = X_test.iloc[:5]
X_nouveau = preprocesseur.transform(nouvelles_donnees)
if hasattr(X_nouveau, "toarray"):
    X_nouveau = X_nouveau.toarray()

with torch.no_grad():
    proba = torch.sigmoid(model(torch.FloatTensor(X_nouveau))).numpy().ravel()

print("Probabilités :", proba.round(3))
print("Classes prédites (0/1) :", (proba >= seuil_final).astype(int))

Probabilités : [0.466 0.288 0.393 0.151 0.189]
Classes prédites (0/1) : [1 0 1 0 0]


## Ensemble de 5 réseaux
Un seul réseau est sensible au hasard (initialisation, ordre des patients). On entraîne donc 5 réseaux identiques avec des graines différentes, puis on fait la moyenne de leurs probabilités. Le seuil est choisi sur la validation à partir de cette moyenne, et le test n'est utilisé qu'à la fin.

In [13]:
from sklearn.metrics import average_precision_score

def entrainer_un_modele(graine, n_epoques=20):
    torch.manual_seed(graine)
    np.random.seed(graine)
    m = Modele(X_tr.shape[1])
    crit = nn.BCEWithLogitsLoss(pos_weight=poids_ajuste)
    opt = optim.AdamW(m.parameters(), lr=0.001, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)
    loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr).reshape(-1, 1)),
        batch_size=64, shuffle=True, drop_last=True)

    meilleur_ap, meilleur_etat_m = -1, None
    for _ in range(n_epoques):
        m.train()
        for bx, by in loader:
            opt.zero_grad()
            crit(m(bx), by).backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            p = torch.sigmoid(m(X_val_tensor)).numpy().ravel()
        ap = average_precision_score(y_val, p)   # plus stable que le F1 pour choisir l'époque
        sched.step(ap)
        if ap > meilleur_ap:
            meilleur_ap, meilleur_etat_m = ap, copy.deepcopy(m.state_dict())
    m.load_state_dict(meilleur_etat_m)
    m.eval()
    return m

def proba_modele(m, X_tenseur):
    with torch.no_grad():
        return torch.sigmoid(m(X_tenseur)).numpy().ravel()

modeles = [entrainer_un_modele(g) for g in range(5)]

proba_val_ens = np.mean([proba_modele(m, X_val_tensor) for m in modeles], axis=0)
seuil_ens, f1_val_ens, precision_val_ens = meilleur_seuil(y_val, proba_val_ens)

proba_test_ens = np.mean([proba_modele(m, X_test_tensor) for m in modeles], axis=0)
prediction_ens = (proba_test_ens >= seuil_ens).astype(int)

print(f'Ensemble : F1 val = {f1_val_ens:.3f} | Seuil = {seuil_ens:.2f}')
print(f'Ensemble : F1 test = {f1_score(y_test, prediction_ens):.3f} | Précision test = {precision_score(y_test, prediction_ens):.3f}')
print(f'Taux de positifs sur le test interne (ensemble) : {prediction_ens.mean():.3f}')

Ensemble : F1 val = 0.358 | Seuil = 0.31
Ensemble : F1 test = 0.315 | Précision test = 0.221
Taux de positifs sur le test interne (ensemble) : 0.198


## Prédiction du fichier de test du professeur
On applique le préprocesseur et l'ensemble de réseaux à un fichier de même structure que `bacteriemie.csv.gz` (avec ou sans les colonnes `ID` et `BloodCulture`), puis on écrit le fichier de rendu avec les colonnes `ID` et `pred` (0/1).

In [14]:
fichier_test = 'bacteriemie_test.csv.gz'   # fichier du professeur
fichier_sortie = 'Groupe2_saoudi_nkrumah.buadii_leyenberger_lamlouh_engin_huang.csv'          # fichier de rendu

def predire_fichier(chemin_entree, chemin_sortie):
    nouveau = pd.read_csv(chemin_entree)
    ids = nouveau['ID'].to_numpy() if 'ID' in nouveau.columns else np.arange(len(nouveau))

    # Même structure que les données d'entraînement
    X_new = nouveau.drop(columns=[c for c in ['BloodCulture', 'ID'] if c in nouveau.columns])
    X_new['SEX'] = X_new['SEX'].astype(object)
    X_new = X_new.reindex(columns=X.columns)

    X_new_prep = preprocesseur.transform(X_new)
    if hasattr(X_new_prep, 'toarray'):
        X_new_prep = X_new_prep.toarray()
    X_new_tenseur = torch.FloatTensor(X_new_prep)

    # Moyenne des 5 réseaux, puis seuil choisi sur la validation
    proba = np.mean([proba_modele(m, X_new_tenseur) for m in modeles], axis=0)
    pred = (proba >= seuil_ens).astype(int)

    sortie = pd.DataFrame({'ID': ids, 'pred': pred})
    assert len(sortie) == len(nouveau), 'Nombre de lignes différent du fichier de test'
    assert set(sortie['pred'].unique()) <= {0, 1}, 'pred doit contenir uniquement 0 et 1'
    sortie.to_csv(chemin_sortie, index=False)
    print(f'{chemin_sortie} écrit : {pred.sum()} positifs sur {len(pred)} ({pred.mean():.1%})')

predire_fichier(fichier_test, fichier_sortie)

Groupe2_saoudi_nkrumah.buadii_leyenberger_lamlouh_engin_huang.csv écrit : 582 positifs sur 2938 (19.8%)
